# 0. Setup
Configure paths and analysis parameters. Data is read from Colab's local `/content/data` when available.

In [ ]:
from pathlib import Path

SEED = 42
MOUNT_DRIVE = False
REPO_URL = "https://github.com/LiliAlania/drone-multi-object-detection.git"
REPO_DIR = Path("/content/drone-multi-object-detection")
DATA_ROOT = Path("/content/data")
TEST_CSV = DATA_ROOT / "test_images.csv"
TRAIN_COCO = DATA_ROOT / "train" / "annotations_coco.json"
VAL_COCO = DATA_ROOT / "val" / "annotations_coco.json"
RESULTS_DIR = REPO_DIR / "results" / "eda"
ANCHORS_PATH = REPO_DIR / "results" / "anchors.json"
FIGURES_DIR = REPO_DIR / "figures" / "eda"
SPLITS_DIR = REPO_DIR / "splits"
IMGSZ_CANDIDATES = [640, 960, 1280, 1536]
ANCHOR_IMGSZ = 960
ANCHOR_K_VALUES = [9, 12]
SUBSAMPLE_STEP = 5
FOLD_COUNT = 5
CONSISTENCY_SAMPLE_FRAMES = 20
TEMPORAL_LAGS = [1, 2, 3, 5, 10, 15, 20, 30]
VISUAL_FRAME_COUNT = 6
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import importlib.util
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
if not (REPO_DIR / "src" / "data_utils.py").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
from src import data_utils as du

In [ ]:
if DATA_ROOT.exists():
    print(f"Using existing dataset directory: {DATA_ROOT}")
else:
    downloader = REPO_DIR / "src" / "get_data.py"
    subprocess.run([sys.executable, str(downloader), "--output-dir", str(DATA_ROOT)], check=True)
required_files = [TRAIN_COCO, VAL_COCO]
missing_files = [path for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f"Expected dataset files not found: {missing_files}")
print(f"Train: {TRAIN_COCO.exists()} | val: {VAL_COCO.exists()} | test CSV: {TEST_CSV.exists()}")

# 1. Load annotations and verify YOLO conversion
Load split metadata, separate target objects from crowd/ignored regions, and compare stored YOLO boxes with normalized COCO boxes.

In [ ]:
train_images_raw, train_annotations_raw = du.load_coco(TRAIN_COCO, "train")
val_images_raw, val_annotations_raw = du.load_coco(VAL_COCO, "val")
train_target_raw, train_ignored_raw = du.split_annotations(train_annotations_raw)
val_target_raw, val_ignored_raw = du.split_annotations(val_annotations_raw)
consistency = {
    "train": du.check_yolo_consistency(TRAIN_COCO, DATA_ROOT / "train", "train", CONSISTENCY_SAMPLE_FRAMES, SEED),
    "val": du.check_yolo_consistency(VAL_COCO, DATA_ROOT / "val", "val", CONSISTENCY_SAMPLE_FRAMES, SEED),
}
print("YOLO/COCO consistency:", consistency)
images, annotations = du.namespace_image_ids(
    [train_images_raw, val_images_raw], [train_annotations_raw, val_annotations_raw]
)
targets, ignored = du.split_annotations(annotations)
train_images = images.loc[images["split"].eq("train")].copy()
val_images = images.loc[images["split"].eq("val")].copy()
train_targets = targets.loc[targets["split"].eq("train")].copy()
val_targets = targets.loc[targets["split"].eq("val")].copy()

**Observations:** TODO

# 2. Dataset overview
Compare clip/frame counts and resolution frequencies for train, validation, and test using annotation metadata and the test CSV.

In [ ]:
test_csv_path = TEST_CSV if TEST_CSV.exists() else None
overview, frames_per_clip, resolution_counts = du.dataset_overview(
    {"train": train_images, "val": val_images}, test_csv_path
)
display(overview)
display(frames_per_clip.groupby("split")["frames"].describe().round(1))
display(resolution_counts.head(20))
resolution_pct = resolution_counts.assign(
    percent=resolution_counts["frames"]
    / resolution_counts.groupby("split")["frames"].transform("sum") * 100
)
resolution_pivot = resolution_pct.pivot(index="resolution", columns="split", values="percent").fillna(0)
ax = resolution_pivot.plot(kind="bar", figsize=(12, 4))
ax.set(title="Image resolution distribution by split", xlabel="Width × height (pixels)", ylabel="Frames (%)")
plt = __import__("matplotlib.pyplot", fromlist=["pyplot"])
plt.tight_layout()
plt.savefig(FIGURES_DIR / "resolution_distribution.png", dpi=140)
plt.show()

**Observations:** TODO

# 3. Class distribution
Inspect target counts and per-clip class shares; examine original VisDrone category IDs and their target/ignored status.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

target_sized = du.add_box_dimensions(targets, target_only=False)
class_counts = (
    target_sized.groupby(["split", "class_name"]).size().rename("objects").reset_index()
)
display(class_counts.pivot(index="class_name", columns="split", values="objects").fillna(0).astype(int))
clip_counts = target_sized.groupby(["clip", "class_name"]).size().unstack(fill_value=0)
clip_shares = clip_counts.div(clip_counts.sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(12, max(4, 0.23 * len(clip_shares))))
image = ax.imshow(clip_shares, aspect="auto", cmap="viridis", vmin=0, vmax=1)
ax.set(title="Target class share by clip", xlabel="Class", ylabel="Clip")
ax.set_xticks(range(len(clip_shares.columns)), clip_shares.columns, rotation=45, ha="right")
ax.set_yticks(range(len(clip_shares.index)), clip_shares.index)
fig.colorbar(image, ax=ax, label="Share of target objects")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "class_share_by_clip.png", dpi=140)
plt.show()
def category_mapping_status(row):
    category_id = pd.to_numeric(row["category_id"], errors="coerce")
    crowd = pd.to_numeric(row["iscrowd"], errors="coerce")
    if pd.notna(crowd) and int(crowd) == 1:
        return "ignored region"
    if pd.notna(category_id) and 1 <= category_id <= len(du.CLASS_NAMES):
        return du.CLASS_NAMES[int(category_id) - 1]
    return "other / non-target"
annotations["mapping_status"] = annotations.apply(category_mapping_status, axis=1)
orig_breakdown = (
    annotations.groupby(["orig_category_id", "mapping_status", "iscrowd"], dropna=False)
    .size().rename("annotations").reset_index()
)
display(orig_breakdown.head(30))

**Observations:** TODO

# 4. Object sizes and input-resolution simulation
Summarize COCO size buckets, pixel dimensions, aspect ratios, per-clip object scale, and the number of pixels available at candidate letterbox sizes.

In [ ]:
size_summary = du.size_bucket_summary(targets)
display(size_summary.head(36))
display(target_sized.groupby("class_name")[["box_w", "box_h", "sqrt_area", "aspect_ratio"]].quantile([0.1, 0.5, 0.9]).round(2).head(30))
clip_size = target_sized.groupby("clip")["sqrt_area"].median().sort_values()
display(clip_size.rename("median_sqrt_area_px").reset_index().head(10))
ax = clip_size.plot(kind="bar", figsize=(14, 4), color="darkcyan")
ax.set(title="Median object scale by clip", xlabel="Clip", ylabel="Median √area (pixels)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "object_size_by_clip.png", dpi=140)
plt.show()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(target_sized["sqrt_area"], bins=50, color="steelblue")
axes[0].set(title="Object square-root area", xlabel="√area (pixels)", ylabel="Objects")
class_order = [name for name in du.CLASS_NAMES if name in target_sized["class_name"].unique()]
axes[1].boxplot(
    [target_sized.loc[target_sized["class_name"].eq(name), "aspect_ratio"].dropna() for name in class_order],
    tick_labels=class_order, showfliers=False
)
axes[1].set(title="Aspect ratio by class", xlabel="Class", ylabel="Width / height")
axes[1].tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "object_size_distributions.png", dpi=140)
plt.show()
resolution_table = du.resized_small_side_shares(targets, images, IMGSZ_CANDIDATES)
display(resolution_table.round(2))

**Observations:** TODO

# 5. Anchor diagnostics
Run seeded IoU-distance k-means on width/height after letterboxing to `ANCHOR_IMGSZ`; anchor values are pixel dimensions at that size.

In [ ]:
anchor_results = [
    du.kmeans_anchors(targets, images, ANCHOR_IMGSZ, k, seed=SEED)
    for k in ANCHOR_K_VALUES
]
du.save_json({"imgsz": ANCHOR_IMGSZ, "results": anchor_results}, ANCHORS_PATH)
display(pd.DataFrame([
    {"k": item["k"], "mean_best_iou": item["mean_best_iou"], "anchors_px": item["anchors"]}
    for item in anchor_results
]))
print(f"Saved anchors to {ANCHORS_PATH}")

**Observations:** TODO

# 6. Objects per frame and density
Inspect the frame-level target count tail against candidate `max_det=500` and DETR query counts of 300 and 500.

In [ ]:
density, density_by_clip, density_stats = du.density_summary(images, targets)
display(pd.DataFrame([density_stats]))
display(density_by_clip.sort_values("max", ascending=False).head(12))
query_check = pd.DataFrame({
    "capacity": [300, 500],
    "frames_over_capacity": [int((density["objects"] > cap).sum()) for cap in [300, 500]],
    "frames_at_or_over_capacity": [int((density["objects"] >= cap).sum()) for cap in [300, 500]],
})
display(query_check)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(density["objects"], bins=40, color="slateblue")
axes[0].set(title="Target objects per frame", xlabel="Objects / frame", ylabel="Frames")
clip_order = density_by_clip.sort_values("median")["clip"]
axes[1].boxplot(
    [density.loc[density["clip"].eq(clip), "objects"] for clip in clip_order],
    tick_labels=clip_order, showfliers=False
)
axes[1].set(title="Frame density by clip", xlabel="Clip", ylabel="Objects / frame")
axes[1].tick_params(axis="x", rotation=90)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "objects_per_frame.png", dpi=140)
plt.show()

**Observations:** TODO

# 7. Ignored regions
Measure per-frame ignored-region incidence and covered area, and count target boxes with any positive-area intersection with an ignored region.

In [ ]:
ignored_by_clip, ignored_stats = du.ignored_region_summary(images, targets, ignored)
display(ignored_stats)
display(ignored_by_clip.sort_values("frames_with_ignored", ascending=False).head(15))

**Observations:** TODO

# 8. Occlusion and truncation
Show annotation counts and normalized distributions for the source occlusion and truncation fields by target class.

In [ ]:
for field in ["occlusion", "truncation"]:
    distribution = (
        targets.groupby(["class_name", field], dropna=False).size()
        .rename("objects").reset_index()
    )
    distribution["class_share"] = distribution["objects"] / distribution.groupby("class_name")["objects"].transform("sum")
    print(field)
    display(distribution.head(32))
    pivot = distribution.pivot(index="class_name", columns=field, values="class_share").fillna(0)
    ax = pivot.plot(kind="bar", stacked=True, figsize=(10, 4), colormap="viridis")
    ax.set(title=f"{field.title()} distribution by class", xlabel="Class", ylabel="Share of class annotations")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"{field}_by_class.png", dpi=140)
    plt.show()

**Observations:** TODO

# 9. Temporal redundancy and frame subsampling
Compare same-track IoU at frame lags and estimate the share of unique clip/track IDs never seen when retaining every kth frame.

In [ ]:
temporal = du.temporal_redundancy(train_images, train_targets, TEMPORAL_LAGS)
display(temporal)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(temporal["step"], temporal["mean_track_iou"], marker="o", label="Mean same-track IoU")
ax.set(title="Temporal box similarity", xlabel="Frame lag k", ylabel="Mean IoU", ylim=(0, 1))
ax2 = ax.twinx()
ax2.plot(temporal["step"], temporal["tracks_never_sampled_pct"], marker="s", color="darkorange", label="Tracks never sampled")
ax2.set_ylabel("Track IDs never sampled (%)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "temporal_redundancy.png", dpi=140)
plt.show()

**Observations:** TODO

# 10. Visual verification
Render six frames selected across clips, including sparse/dense scenes, smallest/largest resolutions, and ignored-region-heavy examples.

In [ ]:
examples = du.select_visual_examples(images, targets, ignored, VISUAL_FRAME_COUNT)
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, (_, frame) in zip(axes.flat, examples.iterrows()):
    frame_targets = targets.loc[targets["image_id"].eq(frame["id"])]
    frame_ignored = ignored.loc[ignored["image_id"].eq(frame["id"])]
    frame_path = du.image_path_for(DATA_ROOT / str(frame["split"]), frame["file_name"])
    du.visualize_frame(frame_path, frame_targets, frame_ignored, ax=ax)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "ground_truth_examples.png", dpi=150, bbox_inches="tight")
plt.show()
display(examples[["split", "clip", "frame_id", "width", "height", "objects", "ignored"]])

**Observations:** TODO

# 11. Sequence-level splits and sampled frame lists
Check train/validation clip separation, summarize five seeded grouped folds over train clips, and save the selected every-kth-frame lists.

In [ ]:
train_clips = set(train_images["clip"].astype(str))
val_clips = set(val_images["clip"].astype(str))
clip_overlap = sorted(train_clips & val_clips)
assert not clip_overlap, f"Train/val clips overlap: {clip_overlap}"
print(f"Train clips: {len(train_clips)} | val clips: {len(val_clips)} | overlap: {len(clip_overlap)}")
folds = du.make_group_folds(train_images, train_targets, FOLD_COUNT, SEED)
du.save_json({"seed": SEED, "n_splits": FOLD_COUNT, "folds": folds}, SPLITS_DIR / "folds.json")
display(pd.DataFrame([
    {"fold": fold["fold"], "val_clips": len(fold["val_clips"]), "train_frames": fold["train_frames"],
     "val_frames": fold["val_frames"], "train_class_counts": fold["train_class_counts"],
     "val_class_counts": fold["val_class_counts"],
     "train_median_sqrt_area": fold["train_median_sqrt_area"],
     "val_median_sqrt_area": fold["val_median_sqrt_area"]}
    for fold in folds
]))
sampled_frames = {
    split: du.subsampled_frame_list(images.loc[images["split"].eq(split)], SUBSAMPLE_STEP)
    for split in ["train", "val"]
}
du.save_json({"step": SUBSAMPLE_STEP, "frames": sampled_frames}, SPLITS_DIR / "subsampled_frames.json")
print({split: len(items) for split, items in sampled_frames.items()})

**Observations:** TODO

# 12. Summary
Persist compact dataset statistics and analysis parameters for later model notebooks.

In [ ]:
summary = {
    "seed": SEED,
    "splits": overview.to_dict("records"),
    "class_counts": class_counts.to_dict("records"),
    "resolution_simulation": resolution_table.to_dict("records"),
    "anchors": {"imgsz": ANCHOR_IMGSZ, "results": anchor_results},
    "density": density_stats,
    "ignored_regions": ignored_stats.to_dict("records"),
    "temporal_redundancy": temporal.to_dict("records"),
    "subsample_step": SUBSAMPLE_STEP,
    "sampled_frame_counts": {split: len(items) for split, items in sampled_frames.items()},
    "train_val_clip_overlap": clip_overlap,
}
du.save_json(summary, RESULTS_DIR / "eda_summary.json")
print(f"Saved summary to {RESULTS_DIR / 'eda_summary.json'}")

**Observations:** TODO